# Рекурентни невронни мрежи

В предишния модул разгледахме богати семантични представяния на текста. Архитектурата, която използваме, улавя обобщеното значение на думите в изречението, но не взема предвид **реда** на думите, тъй като операцията на агрегация след ембедингите премахва тази информация от оригиналния текст. Понеже тези модели не могат да представят реда на думите, те не могат да решават по-сложни или двусмислени задачи като генериране на текст или отговор на въпроси.

За да уловим значението на текстова последователност, ще използваме архитектура на невронна мрежа, наречена **рекурентна невронна мрежа** или RNN. При използване на RNN подаваме изречението през мрежата един токен по един път, а мрежата произвежда някакво **състояние**, което после подаваме отново на мрежата заедно със следващия токен.

![Изображение, показващо пример за генериране с рекурентна невронна мрежа.](../../../../../translated_images/bg/rnn.27f5c29c53d727b5.webp)

При дадена входна последователност от токени $X_0,\dots,X_n$, RNN създава последователност от блокове на невронната мрежа и обучава тази последователност като цяло чрез обратна пропагация. Всеки мрежов блок приема двойката $(X_i,S_i)$ като вход и произвежда $S_{i+1}$ като резултат. Крайната стойност $S_n$ или изход $Y_n$ се подава на линеен класификатор за получаване на резултата. Всички блокове на мрежата споделят едни и същи тегла и се обучават като цяло чрез един пропуск на обратна пропагация.

> Фигурата по-горе показва рекурентна невронна мрежа в разгърната форма (вляво) и в по-компактно рекурентно представяне (вдясно). Важно е да се разбере, че всички RNN клетки имат едни и същи **споделяни тегла**.

Тъй като състоянията $S_0,\dots,S_n$ се подават през мрежата, RNN може да научи последователни зависимости между думите. Например, когато думата *not* се появи някъде в последователността, мрежата може да научи да отрича определени елементи в състоянието.

Всяка RNN клетка съдържа две матрици с тегла: $W_H$ и $W_I$, и биас $b$. При всяка стъпка на RNN, за даден вход $X_i$ и входно състояние $S_i$, изходното състояние се изчислява като $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, където $f$ е активационна функция (често $\tanh$).

> За задачи като генериране на текст (която ще разгледаме в следващия модул) или машинен превод искаме да получаваме и някаква изходна стойност при всяка RNN стъпка. В този случай има и още една матрица $W_O$, и изходът се изчислява като $Y_i=f(W_O\times S_i+b_O)$.

Нека видим как рекурентните невронни мрежи могат да ни помогнат да класифицираме нашия новинарски набор от данни.

> За sandbox средата трябва да изпълним следната клетка, за да сме сигурни, че необходимата библиотека е инсталирана и данните са предварително заредени. Ако работите локално, можете да пропуснете следващата клетка.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

При обучение на големи модели, разпределянето на паметта на GPU може да се превърне в проблем. Може също да е необходимо да експериментираме с различни размери на минипакети, така че данните да се съберат в паметта на GPU, но обучението да е достатъчно бързо. Ако изпълнявате този код на собствена GPU машина, може да експериментирате с настройване на размера на минипакета, за да ускорите обучението.

> **Забележка**: Определени версии на драйвери на NVidia са известни с това, че не освобождават паметта след обучение на модела. В този записвателен бележник изпълняваме няколко примера, и това може да доведе до изчерпване на паметта в определени настройки, особено ако правите свои собствени експерименти като част от същия записвателен бележник. Ако срещнете някакви странни грешки при започване на обучението на модела, може да искате да рестартирате ядрото на записвателния бележник.


In [3]:
batch_size = 16
embed_size = 64

## Прост RNN класификатор

В случая с прост RNN, всяка рекурентна единица е проста линейна мрежа, която приема входен вектор и вектор на състоянието и произвежда нов вектор на състоянието. В Keras това може да се представи чрез слоя `SimpleRNN`.

Въпреки че можем да подаваме директно токени с едно-горещо кодиране към RNN слоя, това не е добра идея заради тяхната висока размерност. Следователно, ще използваме слой за вграждане, за да намалим размерността на векторите на думите, последван от RNN слой и накрая класификатор `Dense`.

> **Забележка**: В случаи, когато размерността не е толкова висока, например при използване на токенизация на ниво символ, може да има смисъл директно да се подават токени с едно-горещо кодиране в RNN клетката.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Забележка:** Тук използваме неличен слой за embedding за простота, но за по-добри резултати можем да използваме предварително обучен слой за embedding с Word2Vec, както е описано в предишната част. Би било полезно упражнение за вас да адаптирате този код да работи с предварително обучени embeddings.

Сега нека обучим нашия RNN. RNN мрежите като цяло са доста трудни за обучение, защото след като RNN клетките се разгънат по дължината на редицата, крайният брой слоеве, участващи в обратното разпространение, е доста голям. Затова трябва да изберем по-ниска скорост на обучение и да обучим мрежата върху по-голям набор от данни, за да постигнем добри резултати. Това може да отнеме доста дълго време, затова предпочитано е използването на GPU.

За да ускорим нещата, ще обучаваме модела RNN само върху заглавия на новини, като пропуснем описанието. Можете да опитате обучение с описанието и да видите дали ще успеете да обучите модела.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Забележка** точността вероятно ще е по-ниска тук, защото обучаваме само на заглавия на новини.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Сега, когато използваме маскиране, можем да обучим модела върху целия набор от заглавия и описания.

> **Бележка**: Забелязали ли сте, че използваме векторизатор, обучен върху новинарските заглавия, а не върху целия текст на статията? Възможно е това да доведе до пренебрегване на някои от токените, затова е по-добре да преквалифицираме векторизатора. Въпреки това, ефектът може да е много малък, затова ще се придържаме към предишния предварително обучен векторизатор за по-голяма простота.


## LSTM: Памет с дълга краткосрочна продължителност

Един от основните проблеми на RNN е **изчезващи градиенти**. RNN може да бъдат доста дълги и може да им е трудно да пропагирират градиентите обратно чак до първия слой на мрежата по време на обратното разпространение. Когато това се случи, мрежата не може да научи връзките между отдалечени символи. Един начин за избягване на този проблем е чрез въвеждане на **експлицитно управление на състоянието** чрез използването на **врати**. Двата най-често срещани архитектури, които въвеждат врати, са **памет с дълга краткосрочна продължителност** (LSTM) и **вратовиден релей единица** (GRU). Тук ще разгледаме LSTM.

![Изображение, показващо примерна клетка на памет с дълга краткосрочна продължителност](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Мрежата LSTM е организирана по начин, подобен на RNN, но има две състояния, които се подават от слой към слой: действителното състояние $c$ и скрития вектор $h$. Във всяка единица скритият вектор $h_{t-1}$ се комбинира с входа $x_t$ и заедно те контролират какво се случва със състоянието $c_t$ и изхода $h_{t}$ чрез **врати**. Всяка врата има сигмоидна активация (изход в диапазона $[0,1]$), която може да се разглежда като побитова маска при умножение със състоянието. LSTM имат следните врати (отляво надясно на горната снимка):
* **врата за забравяне**, която определя кои компоненти от вектора $c_{t-1}$ трябва да забравим и кои да пропуснем. 
* **входна врата**, която определя колко информация от входния вектор и предишния скрит вектор трябва да се включи в състоянието.
* **изходна врата**, която взема новия състоянин вектор и решава кои от неговите компоненти ще се използват за генериране на новия скрит вектор $h_t$.

Компонентите на състоянието $c$ могат да се разглеждат като флагове, които могат да се включват и изключват. Например, когато срещнем името *Алиса* в последователността, предполагаме, че то се отнася за жена и вдигаме флага в състоянието, който казва, че имаме женско съществително в изречението. Когато след това срещнем думите *и Том*, ще вдигнем флага, който казва, че имаме множествено число. Така чрез манипулиране на състоянието можем да следим граматическите свойства на изречението.

> **Бележка**: Ето един страхотен ресурс за разбиране на вътрешното устройство на LSTM: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) от Кристофър Олах.

Въпреки че вътрешната структура на клетката LSTM може да изглежда сложна, Keras скрива тази имплементация вътре в слоя `LSTM`, така че единственото, което трябва да направим в горния пример, е да заменим рекурентния слой:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Забележка**: Обучението на LSTM също е доста бавно и в началото на обучението може да не забележите голямо увеличение на точността. Може да се наложи да продължите обучението за известно време, за да постигнете добра точност.


## Двунасочни и многослойни RNN-и

В нашите досегашни примери, рекурентните мрежи работят от началото на последователността до края ѝ. Това ни се струва естествено, защото следва същата посока, в която четем или слушаме реч. Въпреки това, за сценарии, които изискват случайнен достъп до входната последователност, има повече смисъл да се извършва рекурентното изчисление и в двете посоки. RNN-ите, които позволяват изчисления в двете посоки, се наричат **двунаправени** RNN-и и могат да се създадат като обгърнете рекурентния слой със специален слой `Bidirectonal`.

> **Забележка**: Слойът `Bidirectional` прави две копия на слоя в него и задава свойството `go_backwards` на едно от тези копия на `True`, карайки го да работи в обратната посока по последователността.

Рекурентните мрежи, еднопосочни или двунаправени, улавят модели в последователността и ги съхраняват в векторите на състоянието или ги връщат като изход. Както при свивковите мрежи, можем да изградим друг рекурентен слой след първия, за да уловим по-високо ниво модели, построени от по-ниско ниво модели, извлечени от първия слой. Това ни води до понятието **многослоен RNN**, който се състои от две или повече рекурентни мрежи, където изходът от предишния слой се подава на следващия слой като вход.

![Изображение показващо многослоен long-short-term-memory RNN](../../../../../translated_images/bg/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Снимка от [този прекрасен пост](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) на Фернандо Лопес.*

Keras прави изграждането на тези мрежи лесна задача, защото просто трябва да добавим повече рекурентни слоеве към модела. За всички слоеве, с изключение на последния, трябва да зададем параметъра `return_sequences=True`, защото искаме слоят да връща всички междинни състояния, а не само крайното състояние на рекурентното изчисление.

Нека изградим двуслоен двунаправен LSTM за нашата задача за класификация.

> **Забележка** този код отново отнема доста време за изпълнение, но ни дава най-висока точност, която сме виждали досега. Така че може би си струва да изчакаме и да видим резултата.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## РНМ за други задачи

До момента се съсредоточихме върху използването на РНМ за класифициране на текстови последователности. Но те могат да се справят с много повече задачи, като генериране на текст и машинен превод &mdash; ще разгледаме тези задачи в следващото звено.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
